# Training diagnostics

Read local training logs, compare loss/coverage traces, and summarize runtime.
This notebook does **not** sample policies or retrain models. Run cells in order
using the `gfn` environment after training a run with `--metrics-jsonl`.

**Inputs:** a suite or run directory. **Outputs:** CSV tables, PNG/PDF figures,
source hashes, and a diagnostic report. All `GF_*` command-line controls still work.
See [analysis controls](../../README.md#controls) and the
[implementation](analysis_utils.py) for plotting details.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "code/train.py").is_file() and (p / "requirements.txt").is_file())
for folder in (ROOT / "code", ROOT / "code/notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
from analysis_utils import AnalysisSession, TrainingAnalysis

## 1. Choose input runs

Change the settings below, or leave them as `None` to use environment variables.
The default profile is `smoke`; the README quickstart uses `outputs/quickstart`.
Runs with different scientific settings or source fingerprints stay in separate cohorts.

In [ ]:
# None keeps the GF_* environment setting, or its documented default.
PROFILE = None       # e.g. "smoke", "main", "ablation", "grid-long", "rna-long"
RUN_ROOT = None      # e.g. "outputs/quickstart"; a suite or single run
FIGURE_ROOT = None   # e.g. "figures/my-analysis"; inside this repository
STRICT = None        # True rejects incomplete/incompatible analyses

session = AnalysisSession.from_environment(
    "reproduction.ipynb", root=ROOT, profile=PROFILE, run_root=RUN_ROOT,
    figure_root=FIGURE_ROOT, strict=STRICT, warn_h16_target=False,
)

## 2. Load and inspect the data

The first table has one row per training run; the second previews its logged rounds.
Training-discovered RNA modes and sEH scaffolds are different from fresh-policy
test metrics. Missing values stay missing rather than being reconstructed.

In [ ]:
analysis = TrainingAnalysis(session).load()
display(analysis.training)
display(analysis.metrics.head())

## 3. Plot training traces

Loss uses a trailing 1,000-round mean; max-to-rest ratios retain their spikes.
Per-update logs give the largest update ratio in each round. Older round summaries
are used only for supported single-update runs; multi-update logs without the
required arrays remain missing and are reported. The normalized layout uses seed
470 and can be disabled with `GF_PAPER_LAYOUTS=0`.

In [ ]:
analysis.plot_traces()
analysis.plot_normalized_traces()

## 4. Summarize runtime

The overhead ratio is `100 × reference_flow / training_update`; the denominator
includes reference-flow computation. Wall-clock and CPU+GPU measurements remain
separate. Incomplete seed sets and incompatible profiling settings are reported.

In [ ]:
runtime_summary = analysis.summarize_runtime()
display(runtime_summary)

## 5. Export provenance and check completeness

Inspect `REVIEW_training.csv` for missing inputs or comparability limitations.
`GF_STRICT=1` raises an error if required runs or outputs are missing.

In [ ]:
manifest = analysis.finish()